## **Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## **Load Dataset**

In [ ]:
df_interactions = pd.read_csv("datasets/RAW_interactions.csv")

df_interactions.head()

## **Exploratory Data Analysis on ratings**

1. Some users did not rate, resulting in 0 ratings, even though some of the ratings are positive/ negative.

2. The user ratings are subjective, and despite giving a very good review, the user gave a subpar rating score.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_interactions['rating'], bins=5)
plt.xlabel('User rating')
plt.ylabel('Frequency')
plt.title('Histogram of User Rating')
plt.grid(axis='y', alpha=0.7)
plt.show()

### 0 Ratings

In [ ]:
df_interactions[df_interactions['rating']==0]

### Subjective ratings

In [ ]:
df_interactions[df_interactions['rating'] == 4]

## **Sentiment Analysis**

Applied a pretrained sentiment analysis model, VADER (Fast and efficient but doesn't capture complex linguistic nuances or sarcasm), to analyse the reviews and obtain sentiment scores. A more advanced model, such as BERT, can be utilised once we upgrade to Colab Pro for enhanced computational resources.

In [ ]:
import nltk 
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer # type: ignore

analyzer = SentimentIntensityAnalyzer()

Weighing sentiment score and ratings based on conditions:

*   If there is no review following a rating:
If no rating is present, assign a default rating of 3.
Otherwise, use the existing rating as is.
*   If conflict (sentiment score and ratings are not similar):
Weigh sentiment score less because VADER cannot capture the nuance of food reviews.
Otherwise, weigh sentiment score more.


In [ ]:
def weighted_sentiment(review, rating):

  if not isinstance(review, str):
    if rating == 0:
      return 3 #if no review and rating of 0, then return 3
    return rating  # elif there is no review, then return actual rating

  va = analyzer.polarity_scores(review)['compound']

  avg_sentiment = va

  if avg_sentiment >= 0.7:
    computed_score = 5
  elif avg_sentiment > 0.4:
    computed_score = 4
  elif avg_sentiment > -0.4:
    computed_score = 3
  elif avg_sentiment > -0.7:
    computed_score = 2
  else:
    computed_score = 1

  #if conflict between avg_sentiment and actual sentiment

  if rating == 0 and rating == 0:
    weighted_score = 3
  elif rating == 0:
    weighted_score = computed_score
  elif (rating > 3 and computed_score < 3) or (rating<3 and computed_score >3):
    weighted_score = computed_score * 0.3 + rating * 0.7
  else:
    weighted_score = computed_score * 0.6 + rating * 0.4

  return weighted_score

In [ ]:
# Apply the model to review to get sentiment score
df_interactions['weighted_score'] = df_interactions.apply(lambda x: weighted_sentiment(x['review'], x['rating']), axis=1)

df_interactions.head()

In [ ]:
# Write the file into csv for faster accessing next time with new `weighted_score` column
df_interactions.to_csv("datasets/df_interactions_with_ratings.csv", index=False)

In [ ]:
# use this to load the df_interactions with rating
df_interactions = pd.read_csv("datasets/df_interactions_with_ratings.csv")

df_interactions

In [ ]:
pd.options.display.float_format = '{:.3f}'.format

df_interactions.describe()

In [ ]:
df_interactions['weighted_score'].median()

### New distribution of ratings

After weighing, visualize their distribution: Left skewed because both sentiment score (from reviews) and ratings are left skewed

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_interactions['weighted_score'], bins=50)
plt.xlabel('Weighted Score')
plt.ylabel('Frequency')
plt.title('Histogram of Weighted Scores')
plt.grid(axis='y', alpha=0.7)
plt.show()